In [ ]:
from datetime import datetime
import numpy as np
import pandas as pd
import arviz as az
from os import listdir as ls

from emu_renewal.constants import MOBILITY_SMOOTH_PERIOD, OUTPUTS_PATH, G_MOB_LOCATION_CMAP
from emu_renewal.run import get_mobility_provider
from emu_renewal.renew import MultiStrainModel
from emu_renewal.inputs import get_google_mobility
from emu_renewal.plotting import plot_mob_weights_by_country

In [ ]:
iso3 = "GBR"
prov = get_mobility_provider(iso3, "g_mob")
model = MultiStrainModel(1e6, datetime(2020, 2, 18), datetime(2021, 6, 30), ["alpha"], [datetime(2020, 12, 1)], prov, False)

In [ ]:
# Raw data
mob_data = get_google_mobility("GBR")
mob_data.plot()

In [ ]:
# Smoothed data
smoothed_mob = mob_data.rolling(MOBILITY_SMOOTH_PERIOD, center=True).mean().dropna()
smoothed_mob.plot()

In [ ]:
# Mobility provider dataframe
model.mob_provider.mobility_df.plot()

In [ ]:
# Weighted data if only one weight is positive
location = 3
dummy_weights = np.zeros(6)
dummy_weights[location] = 1.0
pd.Series(prov.get_parameterised_mobility(dummy_weights, 1.0)).plot()

In [ ]:
# Mobility weights from a calibration
job_path = OUTPUTS_PATH / "46693102"
idata = az.from_netcdf(job_path / iso3 / "g_mob/idata_filtered.nc")
idata.posterior["mob_weights"].to_dataframe()

In [ ]:
# Revised weights structure for plotting
weights = idata.posterior["mob_weights"].to_dataframe().unstack("mob_weights_dim_0")
weights.columns = mob_data.columns
weights

In [ ]:
# Check colours and labels as in plotting function
for l in weights.columns:
    colour = G_MOB_LOCATION_CMAP[l]
    label = l.replace("_", " ")
    print(f"location: {label}, colour: {colour}")

In [ ]:
# Check plotting function
weights.median()

In [ ]:
fig = plot_mob_weights_by_country(job_path, ["GBR"])

In [ ]:
# Check input data for all countries has weights in the same order
for iso3 in ls(job_path):
    print(iso3)
    try:
        mob_data = get_google_mobility(iso3)
        print(mob_data.columns)
    except:
        print("no Google data")